# Grade live strikeout snapshots

Concat every `data/ev_snapshots/mlb/strikeouts_*.csv` for a slate, then keep the **closing** line for each game: the last snapshot with `snapshot_at_utc` strictly before that game's first pitch.

A scrape before the 3pm games is the close for those games. A later scrape before the 6pm games is the close only for games that have not started yet. Rows taken after first pitch are ignored.

Settlement uses the starter's final strikeout total. `nb_side` wins when the actual total finishes on that side of `line`. Profit is 1 unit at the American price on that side.


## Knobs

`SLATE_DATE` is the MLB official date. `None` grades every official date in the files. Re-run after games are final to fill strikeouts.


In [5]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("/Users/alexgonzalez/Documents/nba_quant")
SLATE_DATE = "2026-09-22"  # None = every official date in the snapshots
SNAPSHOT_GLOB = "data/ev_snapshots/mlb/strikeouts_*.csv"
FETCH_RESULTS = True  # False = use pitcher_starts already in the store

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.mlb.config import load_config
from src.mlb.markets.odds import american_to_implied, implied_to_decimal
from src.mlb.pipeline.gamelogs import default_http, normalize_player_name
from src.mlb.pipeline.ingest import SCHEDULE_URL
from src.mlb.storage import MlbStore

config = load_config(ROOT / "config" / "mlb.yaml")
store = MlbStore(config)
http = default_http(config) if FETCH_RESULTS else None
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)


In [6]:
FINAL_STATUS = {"final", "game over", "completed early"}
SCHEDULE_META = (
    "scheduled_start_utc",
    "official_date",
    "status",
    "probable_home_pitcher_id",
    "probable_away_pitcher_id",
    "home_p_key",
    "away_p_key",
)


def load_snapshots(root: Path, pattern: str) -> pd.DataFrame:
    paths = sorted(root.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"no snapshots matching {pattern}")
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        frame["path"] = path.name
        frames.append(frame)
    quotes = pd.concat(frames, ignore_index=True)
    quotes["snapshot_at_utc"] = pd.to_datetime(quotes["snapshot_at_utc"], utc=True)
    quotes["line"] = pd.to_numeric(quotes["line"], errors="coerce")
    for col in ("game_pk", "pitcher_id"):
        if col in quotes.columns:
            quotes[col] = pd.to_numeric(quotes[col], errors="coerce").astype("Int64")
        else:
            quotes[col] = pd.Series(pd.NA, index=quotes.index, dtype="Int64")
    if "scheduled_start_utc" in quotes.columns:
        quotes["scheduled_start_utc"] = pd.to_datetime(quotes["scheduled_start_utc"], utc=True)
    else:
        quotes["scheduled_start_utc"] = pd.NaT
    quotes["player_key"] = quotes["player"].map(normalize_player_name)
    split = quotes["game"].fillna("").astype(str).str.split(" @ ", n=1, expand=True)
    quotes["away_name"] = split[0].str.strip()
    quotes["home_name"] = split[1].str.strip() if split.shape[1] > 1 else ""
    quotes["away_key"] = quotes["away_name"].map(normalize_player_name)
    quotes["home_key"] = quotes["home_name"].map(normalize_player_name)
    quotes = quotes.drop_duplicates(
        ["snapshot_at_utc", "book", "player_key", "game", "line", "nb_ev", "game_pk", "pitcher_id"]
    )
    print(f"files {len(paths)} rows {len(quotes)}")
    print(quotes.groupby("path")["snapshot_at_utc"].min().rename("snapshot_at").to_string())
    return quotes


def fetch_schedule(client, start: str, end: str) -> pd.DataFrame:
    payload = client(
        SCHEDULE_URL,
        {"sportId": 1, "startDate": start, "endDate": end, "hydrate": "probablePitcher,team"},
    )
    data = json.loads(payload)
    rows: list[dict] = []
    for day in data.get("dates") or []:
        for game in day.get("games") or []:
            teams = game.get("teams") or {}
            home = teams.get("home") or {}
            away = teams.get("away") or {}
            home_p = home.get("probablePitcher") or {}
            away_p = away.get("probablePitcher") or {}
            rows.append(
                {
                    "game_pk": int(game["gamePk"]),
                    "scheduled_start_utc": game.get("gameDate"),
                    "official_date": str(game.get("officialDate") or day.get("date") or ""),
                    "status": str((game.get("status") or {}).get("detailedState") or ""),
                    "probable_home_pitcher_id": home_p.get("id"),
                    "probable_away_pitcher_id": away_p.get("id"),
                    "probable_home_pitcher": home_p.get("fullName"),
                    "probable_away_pitcher": away_p.get("fullName"),
                    "home_name": (home.get("team") or {}).get("name"),
                    "away_name": (away.get("team") or {}).get("name"),
                }
            )
    slate = pd.DataFrame(rows)
    if slate.empty:
        return slate
    slate["scheduled_start_utc"] = pd.to_datetime(slate["scheduled_start_utc"], utc=True)
    slate["home_key"] = slate["home_name"].map(normalize_player_name)
    slate["away_key"] = slate["away_name"].map(normalize_player_name)
    slate["home_p_key"] = slate["probable_home_pitcher"].map(normalize_player_name)
    slate["away_p_key"] = slate["probable_away_pitcher"].map(normalize_player_name)
    slate["game_pk"] = slate["game_pk"].astype("Int64")
    return slate


def _match_pks(row: dict, slate: pd.DataFrame) -> list[int]:
    if pd.notna(row.get("game_pk")):
        return [int(row["game_pk"])]
    same = slate.loc[slate["away_key"].eq(row["away_key"]) & slate["home_key"].eq(row["home_key"])]
    if same.empty:
        return []
    named = same.loc[same["home_p_key"].eq(row["player_key"]) | same["away_p_key"].eq(row["player_key"])]
    pool = named if not named.empty else same
    et_date = pd.Timestamp(row["snapshot_at_utc"]).tz_convert("America/New_York").strftime("%Y-%m-%d")
    on_slate = pool.loc[pool["official_date"].eq(et_date)]
    pool = on_slate if not on_slate.empty else pool
    return [int(pk) for pk in pool["game_pk"].dropna().unique()]


def attach_schedule(quotes: pd.DataFrame, slate: pd.DataFrame) -> pd.DataFrame:
    lookup = slate.drop_duplicates("game_pk").set_index("game_pk")
    people = store.read_table("id_map")
    name_to_id: dict[str, int] = {}
    if not people.empty and "name" in people.columns:
        keyed = people.assign(player_key=people["name"].map(normalize_player_name))
        keyed = keyed.loc[keyed["player_key"].ne("")].drop_duplicates("player_key")
        name_to_id = {str(row.player_key): int(row.mlb_id) for row in keyed.itertuples(index=False)}
    rows: list[dict] = []
    for rec in quotes.to_dict(orient="records"):
        pks = _match_pks(rec, slate)
        rec["n_game_matches"] = len(pks)
        if len(pks) != 1:
            rec["game_pk"] = pd.NA
            rows.append(rec)
            continue
        rec["game_pk"] = pks[0]
        if pks[0] in lookup.index:
            info = lookup.loc[pks[0]]
            for col in SCHEDULE_META:
                current = rec.get(col)
                if current is None or (not isinstance(current, str) and pd.isna(current)) or current == "":
                    rec[col] = info[col]
        if pd.isna(rec.get("pitcher_id")):
            if rec.get("player_key") == rec.get("home_p_key"):
                rec["pitcher_id"] = rec.get("probable_home_pitcher_id")
            elif rec.get("player_key") == rec.get("away_p_key"):
                rec["pitcher_id"] = rec.get("probable_away_pitcher_id")
            else:
                rec["pitcher_id"] = name_to_id.get(str(rec.get("player_key") or ""))
        rows.append(rec)
    out = pd.DataFrame(rows)
    out["scheduled_start_utc"] = pd.to_datetime(out["scheduled_start_utc"], utc=True)
    out["game_pk"] = pd.to_numeric(out["game_pk"], errors="coerce").astype("Int64")
    out["pitcher_id"] = pd.to_numeric(out["pitcher_id"], errors="coerce").astype("Int64")
    return out


def closing_lines(quotes: pd.DataFrame, slate_date: str | None) -> pd.DataFrame:
    dated = quotes.loc[quotes["scheduled_start_utc"].notna() & quotes["game_pk"].notna()].copy()
    if slate_date:
        dated = dated.loc[dated["official_date"].astype(str).eq(slate_date)]
    pre = dated.loc[dated["snapshot_at_utc"] < dated["scheduled_start_utc"]].copy()
    if pre.empty:
        return pre
    close_at = pre.groupby("game_pk")["snapshot_at_utc"].transform("max")
    close = pre.loc[pre["snapshot_at_utc"].eq(close_at)].copy()
    return close.drop_duplicates(["game_pk", "book", "player_key", "line"])


quotes = load_snapshots(ROOT, SNAPSHOT_GLOB)
span_start = (quotes["snapshot_at_utc"].min().tz_convert("America/New_York") - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
span_end = (quotes["snapshot_at_utc"].max().tz_convert("America/New_York") + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
if http is None:
    raise RuntimeError("set FETCH_RESULTS = True once so the schedule can be pulled")
slate = fetch_schedule(http, span_start, span_end)
quoted = attach_schedule(quotes, slate)
unmatched = quoted.loc[quoted["game_pk"].isna(), ["book", "player", "game", "n_game_matches"]]
print("unmatched", len(unmatched))
if not unmatched.empty:
    print(unmatched.drop_duplicates().head(20).to_string(index=False))
close = closing_lines(quoted, SLATE_DATE)
print(f"closing rows {len(close)} games {close['game_pk'].nunique() if not close.empty else 0}")
if not close.empty:
    picked = (
        close.groupby(["game", "scheduled_start_utc"], as_index=False)["snapshot_at_utc"]
        .max()
        .rename(columns={"snapshot_at_utc": "close_snapshot"})
        .sort_values("scheduled_start_utc")
    )
    print(picked.to_string(index=False))


files 13 rows 779
path
strikeouts_2026-09-23_000020.csv   2026-09-23 00:00:20.623438+00:00
strikeouts_2026-09-23_000817.csv   2026-09-23 00:08:17.084934+00:00
strikeouts_2026-09-23_005145.csv   2026-09-23 00:51:45.347010+00:00
strikeouts_2026-09-23_015354.csv   2026-09-23 01:53:54.716142+00:00
strikeouts_2026-09-23_015850.csv   2026-09-23 01:58:50.318597+00:00
strikeouts_2026-09-23_020057.csv   2026-09-23 02:00:57.845040+00:00
strikeouts_2026-09-23_025724.csv   2026-09-23 02:57:24.084606+00:00
strikeouts_2026-09-24_021544.csv   2026-09-24 02:15:44.093498+00:00
strikeouts_2026-09-24_204021.csv   2026-09-24 20:40:21.971391+00:00
strikeouts_2026-09-24_222106.csv   2026-09-24 22:21:06.950260+00:00
strikeouts_2026-09-25_005921.csv   2026-09-25 00:59:21.306943+00:00
strikeouts_2026-09-25_013548.csv   2026-09-25 01:35:48.918753+00:00
strikeouts_2026-09-25_184245.csv   2026-09-25 18:42:45.340434+00:00
unmatched 0
closing rows 18 games 6
                                   game       scheduled_s

In [7]:
def _boxscore_strikeouts(client, game_pk: int) -> dict[int, int]:
    payload = client(f"https://statsapi.mlb.com/api/v1/game/{int(game_pk)}/boxscore", None)
    box = json.loads(payload)
    found: dict[int, int] = {}
    for side in ("home", "away"):
        players = ((box.get("teams") or {}).get(side) or {}).get("players") or {}
        for player in players.values():
            pitching = (player.get("stats") or {}).get("pitching") or {}
            person = player.get("person") or {}
            if not pitching or person.get("id") is None:
                continue
            found[int(person["id"])] = int(pitching.get("strikeOuts") or 0)
    return found


def attach_results(close: pd.DataFrame, starts: pd.DataFrame, client) -> pd.DataFrame:
    out = close.copy()
    out["strikeouts"] = pd.Series(pd.NA, index=out.index, dtype="Float64")
    if not starts.empty and {"game_pk", "pitcher_id", "strikeouts"}.issubset(starts.columns):
        actual = starts[["game_pk", "pitcher_id", "strikeouts"]].drop_duplicates(
            ["game_pk", "pitcher_id"], keep="last"
        )
        actual["game_pk"] = pd.to_numeric(actual["game_pk"], errors="coerce").astype("Int64")
        actual["pitcher_id"] = pd.to_numeric(actual["pitcher_id"], errors="coerce").astype("Int64")
        out = out.drop(columns=["strikeouts"]).merge(actual, on=["game_pk", "pitcher_id"], how="left")
    if client is not None and not out.empty:
        final_pks = (
            out.loc[
                out["status"].astype(str).str.lower().isin(FINAL_STATUS) & out["strikeouts"].isna(),
                "game_pk",
            ]
            .dropna()
            .astype(int)
            .unique()
        )
        for game_pk in final_pks:
            try:
                found = _boxscore_strikeouts(client, int(game_pk))
            except Exception as exc:
                print(f"boxscore {game_pk} failed: {exc}")
                continue
            mask = out["game_pk"].eq(game_pk) & out["strikeouts"].isna()
            out.loc[mask, "strikeouts"] = out.loc[mask, "pitcher_id"].map(found)
    return out


def _side_price(row: pd.Series) -> float:
    price = row.over if row.nb_side == "over" else row.under
    return float(price) if pd.notna(price) else float("nan")


def grade(close: pd.DataFrame) -> pd.DataFrame:
    out = close.copy()
    out["price"] = out.apply(_side_price, axis=1)
    out["p_side"] = np.where(out["nb_side"].eq("over"), out["p_over"], out["p_under"])
    k = pd.to_numeric(out["strikeouts"], errors="coerce")
    is_final = out["status"].astype(str).str.lower().isin(FINAL_STATUS)
    settled = is_final & k.notna()
    won = np.where(out["nb_side"].eq("over"), k > out["line"], k < out["line"])
    push = k.eq(out["line"])
    out["result"] = np.where(
        settled & push,
        "push",
        np.where(settled & won, "win", np.where(settled, "loss", np.where(is_final, "no_start", "pending"))),
    )
    profit: list[float] = []
    for result, price in zip(out["result"], out["price"], strict=True):
        if result in {"pending", "no_start"} or not np.isfinite(price):
            profit.append(float("nan"))
        elif result == "push":
            profit.append(0.0)
        elif result == "win":
            decimal = float(implied_to_decimal(float(american_to_implied(float(price)))))
            profit.append(decimal - 1.0)
        else:
            profit.append(-1.0)
    out["profit"] = profit
    return out


starts = store.read_table("pitcher_starts")
graded = grade(attach_results(close, starts, http))
show = [
    "book",
    "player",
    "game",
    "line",
    "nb_side",
    "price",
    "nb_ev",
    "p_side",
    "strikeouts",
    "result",
    "profit",
    "snapshot_at_utc",
    "scheduled_start_utc",
]
print(graded.sort_values(["scheduled_start_utc", "nb_ev"], ascending=[True, False])[show].to_string(index=False))


      book          player                                    game  line nb_side    price  nb_ev  p_side  strikeouts result  profit                  snapshot_at_utc       scheduled_start_utc
  pinnacle  MacKenzie Gore           New York Mets @ Texas Rangers 4.500   under -126.000  0.150   0.641       2.000    win   0.794 2026-09-23 00:00:20.623438+00:00 2026-09-23 00:05:00+00:00
draftkings  MacKenzie Gore           New York Mets @ Texas Rangers 4.500   under      NaN -0.243   0.641       2.000    win     NaN 2026-09-23 00:00:20.623438+00:00 2026-09-23 00:05:00+00:00
draftkings     Sean Manaea           New York Mets @ Texas Rangers 4.500   under      NaN -0.554   0.739       9.000   loss     NaN 2026-09-23 00:00:20.623438+00:00 2026-09-23 00:05:00+00:00
prizepicks  MacKenzie Gore           New York Mets @ Texas Rangers 2.500   under      NaN    NaN   0.260       2.000    win     NaN 2026-09-23 00:00:20.623438+00:00 2026-09-23 00:05:00+00:00
prizepicks     Sean Manaea           New York

## How the closes did

Pending games have not gone final. `no_start` means the quoted pitcher did not throw in a final game. The `+EV` table is the side the card would have bet (`nb_ev > 0`) at that game's close.


In [8]:
settled = graded.loc[graded["result"].isin(["win", "loss", "push"])].copy()
positive = settled.loc[settled["nb_ev"] > 0].copy()


def _summarize(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    out = frame.groupby("book", as_index=False).agg(
        bets=("result", "size"),
        wins=("result", lambda s: int((s == "win").sum())),
        losses=("result", lambda s: int((s == "loss").sum())),
        pushes=("result", lambda s: int((s == "push").sum())),
        profit=("profit", "sum"),
        mean_p=("p_side", "mean"),
    )
    decided = out["wins"] + out["losses"]
    out["hit_rate"] = out["wins"] / decided.where(decided > 0)
    return out.sort_values("profit", ascending=False)


print("settled", len(settled), "pending", int((graded["result"] == "pending").sum()), "no_start", int((graded["result"] == "no_start").sum()))
print("\nall closing lines")
print(_summarize(settled).to_string(index=False) if not settled.empty else "(none final yet)")
print("\n+EV closes only")
print(_summarize(positive).to_string(index=False) if not positive.empty else "(none final yet)")


settled 18 pending 0 no_start 0

all closing lines
      book  bets  wins  losses  pushes  profit  mean_p  hit_rate
     novig     2     2       0       0   0.870   0.653     1.000
  pinnacle     1     1       0       0   0.794   0.641     1.000
draftkings    11     7       4       0   0.000   0.657     0.636
prizepicks     2     1       1       0   0.000   0.410     0.500
  underdog     2     0       2       0  -2.000   0.648     0.000

+EV closes only
      book  bets  wins  losses  pushes  profit  mean_p  hit_rate
     novig     1     1       0       0   0.870   0.568     1.000
  pinnacle     1     1       0       0   0.794   0.641     1.000
draftkings     1     1       0       0   0.000   0.207     1.000
  underdog     2     0       2       0  -2.000   0.648     0.000
